# Circulation Graph
Spatial connectivity through apertures (doors and windows).  
Two rooms are connected only when a door or window sits on their shared face.

## 1. Import Libraries

In [1]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Wire import Wire
from topologicpy.Face import Face
from topologicpy.Shell import Shell
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper
from topologicpy.Color import Color

e:\softwares-4\graph-ml\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(Helper.Version())

The version that you are using (0.9.21) is EQUAL TO the latest version available on PyPI.


In [3]:
renderer = "vscode"

## 2. Import OBJ Files

In [4]:
objects = Topology.ByOBJPath(r"E:\softwares-4\graph-ml\assign-01\geometry\box-house-rooms.obj", selfMerge=True)
print("Objects:", objects)

Objects: [<topologic_core.Cluster object at 0x000002601E02D6B0>]


In [5]:
doors   = Topology.ByOBJPath(r"E:\softwares-4\graph-ml\assign-01\geometry\box-house-doors.obj",   selfMerge=True)
windows = Topology.ByOBJPath(r"E:\softwares-4\graph-ml\assign-01\geometry\box-house-windows.obj", selfMerge=True)

# Flatten clusters to individual faces and colour-code
aperture_faces = []
for ap in doors:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color"], ["brown"]))
        aperture_faces.append(f)
for ap in windows:
    for f in (Topology.Faces(ap) or [ap]):
        Topology.SetDictionary(f, Dictionary.ByKeysValues(["color"], ["cyan"]))
        aperture_faces.append(f)

print("Aperture faces:", len(aperture_faces))

Aperture faces: 36


## 3. Extract Cells and Build CellComplex

In [6]:
cells = Topology.Cells(objects[0])
print("Number of cells:", len(cells))

cc = CellComplex.ByCells(cells)
cc = Topology.RemoveCoplanarFaces(cc)
cc = Topology.RemoveCollinearEdges(cc)
print("CellComplex:", cc)

Number of cells: 19
CellComplex: <topologic_core.CellComplex object at 0x000002601E066870>


## 4. Show Geometry

In [15]:
ap_cluster = Cluster.ByTopologies(aperture_faces)
Topology.Show(cc, ap_cluster,
              faceColorKey="color",
              faceOpacity=0.5,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 5. Add Apertures to CellComplex

In [8]:
cc = Topology.AddApertures(cc, aperture_faces, subTopologyType="face")
print("Apertures added:", len(aperture_faces))

Apertures added: 36


## 6. Build the Circulation Graph
`useApertures=True` — cells connect only through a door or window on their shared face.

In [17]:
g_circ = Graph.ByTopology(cc,
                           direct=False,
                           viaSharedApertures=True,
                           toExteriorApertures=False)
print("Vertices:", len(Graph.Vertices(g_circ)))
print("Edges:",    len(Graph.Edges(g_circ)))

Vertices: 39
Edges: 40


## 7. Assign Visual Attributes

In [18]:
for v in Graph.Vertices(g_circ):
    d = Dictionary.ByKeysValues(["size", "color"], [18, "red"])
    Topology.SetDictionary(v, d)

for e in Graph.Edges(g_circ):
    d = Dictionary.ByKeysValues(["width", "color"], [4, "black"])
    Topology.SetDictionary(e, d)

## 8. Show Circulation Graph

In [19]:
Topology.Show(g_circ,
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)

## 9. Show Circulation Graph Overlaid on Geometry

In [13]:
ap_cluster = Cluster.ByTopologies(aperture_faces)
Topology.Show(cc, ap_cluster, g_circ,
              faceColorKey="color",
              vertexSizeKey="size",
              vertexColorKey="color",
              edgeWidthKey="width",
              edgeColorKey="color",
              faceOpacity=0.4,
              backgroundColor="white",
              width=700,
              height=500,
              renderer=renderer)